# Demo: orquestador multi-agente (Supervisor → Investigador → Analista → Síntesis)

Este notebook corre el grafo de `graph.py` **contra servicios reales**: Gemini (endpoint compatible con OpenAI) para los cuatro roles y el índice Pinecone `pre-entrega-4` (namespace `caudal-docs`) de la Pre-entrega 4 para la búsqueda híbrida del Investigador.

La pregunta mezcla dos dominios a propósito: **datos documentales** (qué dice la documentación de Caudal sobre el tamaño de lote) y **cálculo** (cuántos lotes y cuánto tiempo). Así el Supervisor tiene que delegar en el Investigador, después en el Analista, validar con su rúbrica y recién ahí cerrar con la síntesis.

Las keys se leen del `.env` y nunca se imprimen.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(".env")
# Un modelo por rol para repartir la cuota diaria del free tier (20 pedidos/día por modelo)
# y un tope de pedidos por minuto compartido. setdefault: si ya están en el entorno, mandan esos.
os.environ.setdefault("LLM_RPM", "4")
os.environ.setdefault("SUPERVISOR_MODEL", "gemini-3-flash-preview")
os.environ.setdefault("RESEARCH_MODEL", "gemini-3.5-flash-lite")
os.environ.setdefault("ANALYST_MODEL", "gemini-3-flash-preview")
os.environ.setdefault("SYNTH_MODEL", "gemini-3.1-flash-lite")

from main import PREGUNTA_DEMO, build_from_env, ejecutar, guardar_traza

graph, modelos = build_from_env()
modelos

{'supervisor': 'gemini-3-flash-preview',
 'research': 'gemini-3.5-flash-lite',
 'analyst': 'gemini-3-flash-preview',
 'synth': 'gemini-3.1-flash-lite'}

## 1. El grafo

Diagrama generado desde el grafo compilado con `graph.get_graph().draw_mermaid()`. Las flechas punteadas salen del Supervisor: son la arista condicional `route_from_supervisor`, cuyo retorno `Literal["researcher", "analyst", "synthesizer"]` define los destinos.

In [2]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	researcher(researcher)
	analyst(analyst)
	synthesizer(synthesizer)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	analyst --> supervisor;
	researcher --> supervisor;
	supervisor -.-> analyst;
	supervisor -.-> researcher;
	supervisor -.-> synthesizer;
	synthesizer --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. La consulta

In [3]:
print(PREGUNTA_DEMO)

Según la documentación de Caudal, ¿cuál es el tamaño de lote (tamano_lote) por defecto y cuál se recomienda para cargas de ingesta masiva? Si tengo que procesar 2.000.000 de eventos con 4 workers y cada worker procesa 5 lotes por segundo, ¿cuántos lotes se forman y cuánto tardaría con cada uno de esos dos tamaños de lote?


## 3. Ejecución con traza de delegación

Cada bloque es lo que devolvió un nodo apenas terminó (`astream(stream_mode="updates")`): las decisiones del Supervisor (destino, razón, rúbrica, instrucción), y las contribuciones de cada especialista con sus tool calls reales y sus fuentes.

In [4]:
final, traza = await ejecutar(graph, PREGUNTA_DEMO, verbose=True)
guardar_traza(PREGUNTA_DEMO, final, traza, modelos)
print("Ruta:", " -> ".join(p["nodo"] for p in traza))

[  18.6s] SUPERVISOR (turno 1) -> researcher
  razón: Se requiere obtener los valores de tamano_lote (por defecto y recomendado) de la documentación antes de realizar los cálculos.
  rúbrica: cobertura=✗, evidencia=✗, calculos=✗, consistencia=✓
  instrucción: Busca en la documentación oficial de la librería Caudal el valor por defecto del parámetro 'tamano_lote' y el valor recomendado específicamente para escenarios de ingesta masiva. Asegúrate de citar la fuente de la información.



/Users/flaviasendino/Documents/Proyectos Claude/Curso AI Engineering/Pre Entrega 6/.venv/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SupervisorDecision(rubric...te de la información."), input_type=SupervisorDecision])
  return self.__pydantic_serializer__.to_python(


[  62.6s] RESEARCHER (turno 1, estado ok)
  tarea: Busca en la documentación oficial de la librería Caudal el valor por defecto del parámetro 'tamano_lote' y el valor recomendado específicamente para escenarios de ingesta masiva. Asegúrate de citar la fuente de la información.
  herramienta: buscar_documentacion({"consulta": "tamano_lote valor por defecto ingesta masiva recomendada"})
  herramienta: buscar_documentacion({"consulta": "tamano_lote ingesta masiva recomendada caudal-rendimiento"})
  fuentes: caudal-rendimiento › Backpressure y tamaño de lote, caudal-instalacion › Configuración con CaudalConfig, caudal-rendimiento › Paralelismo y workers, caudal-seguridad › Auditoría y datos personales, caudal-errores › Diagnóstico de problemas de rendimiento y memoria, caudal-api-flujos › La clase Flujo y sus operadores
  contribución: * Valor por defecto de `tamano_lote`: 500 [caudal-instalacion › Configuración con CaudalConfig].
    * Valor recomendado para escenarios de ingesta masiva (

[  86.7s] SUPERVISOR (turno 2) -> analyst
  razón: El investigador ya obtuvo los valores de tamano_lote (500 y 5000). Ahora el analista debe realizar los cálculos de cantidad de lotes y tiempo total solicitados en la pregunta.
  rúbrica: cobertura=✗, evidencia=✓, calculos=✗, consistencia=✓
  instrucción: Calculá la cantidad de lotes y el tiempo total de procesamiento (en segundos) para dos escenarios: 1) tamano_lote = 500 y 2) tamano_lote = 5000. Datos de entrada: total de eventos = 2.000.000, cantidad de workers = 4, velocidad de procesamiento = 5 lotes por segundo por cada worker. Utilizá herramientas aritméticas para asegurar la precisión.



/Users/flaviasendino/Documents/Proyectos Claude/Curso AI Engineering/Pre Entrega 6/.venv/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SupervisorDecision(rubric...segurar la precisión.'), input_type=SupervisorDecision])
  return self.__pydantic_serializer__.to_python(


[ 155.7s] ANALYST (turno 2, estado ok)
  tarea: Calculá la cantidad de lotes y el tiempo total de procesamiento (en segundos) para dos escenarios: 1) tamano_lote = 500 y 2) tamano_lote = 5000. Datos de entrada: total de eventos = 2.000.000, cantidad de workers = 4, velocidad de procesamiento = 5 lotes por segundo por cada worker. Utilizá herramientas aritméticas para asegurar la precisión.
  herramienta: estimar_tiempo_procesamiento({"lotes_por_segundo_por_worker": 5, "workers": 4, "tamano_lote": 500, "registros": 2000000})
  herramienta: estimar_tiempo_procesamiento({"tamano_lote": 5000, "workers": 4, "registros": 2000000, "lotes_por_segundo_por_worker": 5})
  fuentes: estimar_tiempo_procesamiento
  contribución: Aquí tenés los cálculos para los dos escenarios solicitados:
    
    **Escenario 1 (tamano_lote = 500)**
    *   **Cantidad de lotes:** 4.000 (calculado como 2.000.000 / 500).
    *   **Tiempo total:** 200 segundos (calculado como 4.000 lotes / (5 lotes/seg * 4 workers)).
  

[ 171.7s] SUPERVISOR (turno 3) -> synthesizer
  razón: Se han obtenido los valores de la documentación y se han realizado los cálculos solicitados para ambos escenarios de forma consistente.
  rúbrica: cobertura=✓, evidencia=✓, calculos=✓, consistencia=✓



/Users/flaviasendino/Documents/Proyectos Claude/Curso AI Engineering/Pre Entrega 6/.venv/lib/python3.12/site-packages/pydantic/main.py:520: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=SupervisorDecision(rubric...FINISH', instruccion=''), input_type=SupervisorDecision])
  return self.__pydantic_serializer__.to_python(


[ 240.5s] SYNTHESIZER
  respuesta final:
    Basado en la documentación de Caudal y los cálculos realizados por nuestro equipo de análisis, aquí tenés la información solicitada:
    
    ### Configuración de `tamano_lote`
    *   **Valor por defecto:** 500 eventos [caudal-instalacion › Configuración con CaudalConfig].
    *   **Recomendación para ingesta masiva:** 5.000 eventos o más, con el fin de aumentar el *throughput* y amortizar la serialización y escrituras en red [caudal-rendimiento › Backpressure y tamaño de lote].
    
    ### Análisis de procesamiento (2.000.000 de eventos)
    Para el cálculo, se consideró una capacidad total de procesamiento de 20 lotes por segundo (4 workers × 5 lotes/segundo por worker) [estimar_tiempo_procesamiento].
    
    **1. Escenario con tamaño de lote de 500:**
    *   **Cantidad de lotes:** 4.000 (2.000.000 / 500) [estimar_tiempo_procesamiento].
    *   **Tiempo total:** 200 segundos (4.000 lotes / 20 lotes por segundo) [estimar_tiempo_procesam

Ruta: supervisor -> researcher -> supervisor -> analyst -> supervisor -> synthesizer


## 4. Estado compartido: quién aportó qué

`contributions` es la lista acumulada (reducer `operator.add`) del `OrchestratorState`.

In [5]:
for i, c in enumerate(final["contributions"], start=1):
    print(f"#{i} {c['agente']:<10} turno {c['paso']}  estado={c['estado']}")
    print(f"   herramientas: {c['herramientas']}")
    print(f"   fuentes:      {c['fuentes']}")
print()
for d in final["decisiones"]:
    print(f"supervisor turno {d['paso']}: {d['next']:<10} rúbrica={d['rubrica']}")
print("\ntask_completed:", final["task_completed"], "| cierre_forzado:", final["cierre_forzado"])

#1 researcher turno 1  estado=ok
   herramientas: ['buscar_documentacion', 'buscar_documentacion']
   fuentes:      ['caudal-rendimiento › Backpressure y tamaño de lote', 'caudal-instalacion › Configuración con CaudalConfig', 'caudal-rendimiento › Paralelismo y workers', 'caudal-seguridad › Auditoría y datos personales', 'caudal-errores › Diagnóstico de problemas de rendimiento y memoria', 'caudal-api-flujos › La clase Flujo y sus operadores']
#2 analyst    turno 2  estado=ok
   herramientas: ['estimar_tiempo_procesamiento', 'estimar_tiempo_procesamiento']
   fuentes:      ['estimar_tiempo_procesamiento']

supervisor turno 1: researcher rúbrica={'cobertura': False, 'evidencia': False, 'calculos': False, 'consistencia': True}
supervisor turno 2: analyst    rúbrica={'cobertura': False, 'evidencia': True, 'calculos': False, 'consistencia': True}
supervisor turno 3: FINISH     rúbrica={'cobertura': True, 'evidencia': True, 'calculos': True, 'consistencia': True}

task_completed: True | cie

## 5. Respuesta final (nodo `synthesizer`)

In [6]:
from IPython.display import Markdown
Markdown(final["final_answer"])

Basado en la documentación de Caudal y los cálculos realizados por nuestro equipo de análisis, aquí tenés la información solicitada:

### Configuración de `tamano_lote`
*   **Valor por defecto:** 500 eventos [caudal-instalacion › Configuración con CaudalConfig].
*   **Recomendación para ingesta masiva:** 5.000 eventos o más, con el fin de aumentar el *throughput* y amortizar la serialización y escrituras en red [caudal-rendimiento › Backpressure y tamaño de lote].

### Análisis de procesamiento (2.000.000 de eventos)
Para el cálculo, se consideró una capacidad total de procesamiento de 20 lotes por segundo (4 workers × 5 lotes/segundo por worker) [estimar_tiempo_procesamiento].

**1. Escenario con tamaño de lote de 500:**
*   **Cantidad de lotes:** 4.000 (2.000.000 / 500) [estimar_tiempo_procesamiento].
*   **Tiempo total:** 200 segundos (4.000 lotes / 20 lotes por segundo) [estimar_tiempo_procesamiento].

**2. Escenario con tamaño de lote de 5.000:**
*   **Cantidad de lotes:** 400 (2.000.000 / 5.000) [estimar_tiempo_procesamiento].
*   **Tiempo total:** 20 segundos (400 lotes / 20 lotes por segundo) [estimar_tiempo_procesamiento].

## 6. Chequeos de la corrida

La ruta tiene que pasar por el Investigador (con fuentes reales de Pinecone) antes que por el Analista (con una herramienta de cálculo), y terminar en la síntesis.

In [7]:
ruta = [p["nodo"] for p in traza]
assert ruta[0] == "supervisor" and ruta[-1] == "synthesizer"
assert ruta.index("researcher") < ruta.index("analyst")
inv = next(c for c in final["contributions"] if c["agente"] == "researcher")
ana = next(c for c in final["contributions"] if c["agente"] == "analyst")
assert inv["fuentes"] and "buscar_documentacion" in inv["herramientas"]
assert set(ana["herramientas"]) & {"calcular", "estimar_tiempo_procesamiento"}
assert final["task_completed"]
print("OK: supervisor -> researcher -> analyst -> synthesizer, con fuentes y cálculos por herramienta")

OK: supervisor -> researcher -> analyst -> synthesizer, con fuentes y cálculos por herramienta
